In [1]:
from pathlib import Path
import tempfile

from embedding_lab.chunking import Chunk
from embedding_lab.vector_store import ChromaVectorStore


class KeywordEmbeddings:
    def _vector(self, text: str) -> list[float]:
        if "幕墙" in text:
            return [1.0, 0.0]
        if "楼梯" in text:
            return [0.0, 1.0]
        return [0.5, 0.5]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._vector(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._vector(text)

In [2]:
def make_chunk(
    chunk_id: str,
    text: str,
    source: str,
    doc_scope: str,
) -> Chunk:
    return Chunk(
        id=chunk_id,
        text=text,
        metadata={
            "namespace": "query_test",
            "source": source,
            "heading": source,
            "doc_scope": doc_scope,
        },
    )


chunks = [
    make_chunk(
        "wall-001",
        "幕墙需要通过 parentWall 挂接",
        "components/walls.md",
        "generation",
    ),
    make_chunk(
        "stair-001",
        "楼梯连接上下楼层",
        "components/stairs.md",
        "generation",
    ),
    make_chunk(
        "readme-001",
        "幕墙和楼梯文档目录",
        "README.md",
        "index",
    ),
]

In [3]:
persist_dir = Path(tempfile.mkdtemp(prefix="studyagent-query-"))
store = ChromaVectorStore(
    persist_dir=persist_dir,
    collection_name="query_demo",
    namespace="query_test",
    index_signature="keyword-model-v1",
)
embeddings = KeywordEmbeddings()

store.sync(chunks, embeddings)
print("索引状态：", store.status())

索引状态： {'persist_dir': 'C:\\Users\\ADMINI~1\\AppData\\Local\\Temp\\studyagent-query-au3p8htz', 'collection': 'query_demo', 'count': 3, 'metadata': {'namespace': 'query_test', 'hnsw:space': 'cosine', 'course': 'studyAgent/07-embedding-basics', 'index_signature': 'keyword-model-v1'}}


In [4]:
hits = store.query("幕墙应该怎样挂接？", embeddings, k=3)

assert hits
assert hits[0].id == "wall-001"
assert all(hit.metadata["doc_scope"] == "generation" for hit in hits)
assert all(hit.id != "readme-001" for hit in hits)

for rank, hit in enumerate(hits, start=1):
    print(f"Top {rank} | distance={hit.distance}")
    print("source:", hit.metadata["source"])
    print("text:", hit.text)

Top 1 | distance=0.0
source: components/walls.md
text: 幕墙需要通过 parentWall 挂接
Top 2 | distance=1.0
source: components/stairs.md
text: 楼梯连接上下楼层


In [5]:
for k in (1, 2, 10):
    current_hits = store.query("幕墙应该怎样挂接？", embeddings, k=k)
    print(f"k={k}, 返回数量={len(current_hits)}")
    print([hit.id for hit in current_hits])

k=1, 返回数量=1
['wall-001']
k=2, 返回数量=2
['wall-001', 'stair-001']
k=10, 返回数量=2
['wall-001', 'stair-001']


In [6]:
class WrongDimensionEmbeddings(KeywordEmbeddings):
    def embed_query(self, text: str) -> list[float]:
        return [1.0, 0.0, 0.0]


try:
    store.query(
        "幕墙应该怎样挂接？",
        WrongDimensionEmbeddings(),
        k=1,
    )
except Exception as exc:
    print("预期维度错误：", type(exc).__name__, exc)
else:
    raise AssertionError("三维问题向量不应该查询二维集合")

预期维度错误： InvalidArgumentError Collection expecting embedding with dimension of 2, got 3
